## What Are Guardrails?
Guardrails are safety mechanisms that validate and filter content at key points in your agent’s execution. In LangChain, they are implemented as middleware that intercepts execution at three levels:

Before the agent starts (input guardrails) -- Block harmful requests, detect PII, enforce authentication, or apply rate limiting before any LLM processing happens. This saves cost because blocked requests never hit your model.

After the agent completes (output guardrails) -- Validate the final response before the user sees it. Check for safety, add compliance disclaimers, remove sensitive information that slipped through, or enforce quality standards.

Around model and tool calls -- Intercept specific tool calls to require human approval, redact PII from tool inputs/outputs, or apply business rules to specific operations.

Common use cases include PII leakage prevention (redacting emails and credit cards before logging), prompt injection blocking (detecting adversarial inputs), harmful content filtering (blocking dangerous requests), business rule enforcement (requiring approval for financial operations), and output quality validation (ensuring responses meet safety standards).

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## Section 1: Two Approaches to Guardrails

Before building anything, you need to understand the two fundamental approaches to guardrails.

## Deterministic Guardrails

These are rule-based checks: regex patterns, keyword matching, explicit logic. They are fast, predictable, and cost-effective -- but they can miss nuanced violations.

In [5]:
import re

def deterministic_guardrail(text: str) -> str:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the captial of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
BLOCKED: How do I hack into a database?
ALLOWED: What is the captial of France?
BLOCKED: Explain how malware spreads


Fast and cheap -- but notice it would also block “Explain how companies protect against malware,” which is a perfectly legitimate question. Keyword matching has no understanding of intent.

# Model-Based Guardrails
These use an LLM or classifier for semantic understanding. They catch subtle and nuanced issues that keyword matching misses -- but they are slower and more expensive.



In [ ]:
from langchain_openai import ChatOpenAI

def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)
    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()
    
print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}: {inp}")

The model-based approach can understand intent. It knows that “Explain how malware spreads” in an educational context might be safe, while “How do I create malware” is not. This nuance is impossible with keyword matching alone.

The golden rule: use deterministic guardrails first (cheap, fast), then model-based guardrails second (expensive, thorough). This way, obviously bad requests get caught early without wasting LLM calls.



## Section 2: Built-in PII Detection Middleware

LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information. It supports multiple PII types -- email, credit card, IP address, MAC address, URL, and API keys -- and multiple strategies for handling them.

redact -- Replaces with [REDACTED_EMAIL] mask -- Replaces with ****-****-****-1234 hash -- Replaces with a hash like a8f5f167... block -- Raises an exception, stopping execution entirely

Here is how to set it up:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# Create agent with PII Middleware
agent = create_agent(
    model="gpt-4o",
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Three layers of PII protection in one agent. Emails get redacted, credit cards get masked, and API keys trigger a hard block. Let us test each scenario.

## Test 1: PII Redaction in Action


In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "My email is john.doe@example.com and my card is"
            "5105-1051-0510-5100. Can you help me?"
        )
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

The agent receives the message with the email replaced by [REDACTED_EMAIL] and the credit card replaced by ****-****-****-5100. The actual PII never reaches the model, the logs, or any downstream system.

## Test 2: API Key Blocking

In [ ]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
except Exception as e:
    print(f"Blocked as expected: {e}")

When an API key is detected, the block strategy raises an exception immediately. The request never reaches the model. This is critical for preventing accidental secret leakage in production systems.

## Section 3: Built-in Human-in-the-Loop Middleware

Some operations are too sensitive for an AI to execute autonomously. Sending emails, deleting database records, processing financial transactions -- these need a human to approve.

LangChain’s HumanInTheLoopMiddleware pauses agent execution before sensitive tool calls and waits for human approval. It requires a checkpointer for state persistence across the interrupt.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

# Create agent with HITL middleware
hitl_agent = create_agent(
    model = "gpt-4o",
    tools = [search_web, send_email, delete_records],
    middleware = [
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_email" : True,    # Require approval
                "delete_records" : True,    # Require approval
                "search_web": False,        # Auto approve
            }
        ),
    ],
    checkpointer = InMemorySaver(), # Required for state persistence
)

The interrupt_on dictionary is the key configuration. You specify exactly which tools need human approval and which can auto-execute. Web searches are harmless -- let them through. Emails and database deletions? Those need a human in the loop.

# Approval Flow

In [ ]:
# Step 1: Invoke -- agent will pause before send_mail
config = {"configuable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config = config
)

print("=== Agent paused -- awaiting human approval ===")

The agent plans the email, prepares the tool call, and then pauses. It does not execute. The state is saved to the checkpointer.

In [ ]:
# Step 2: Human reviews and APPROVES
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config  # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

Using the same thread_id, the human sends an approve command, and the agent resumes execution from where it paused.

## Rejection Flow

In [ ]:
# Alternative -- Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

When a human rejects, the agent receives the rejection reason and responds accordingly. The dangerous operation never executes.

## Section 4: Custom Before-Agent Guardrail (Input Filter)
The built-in middleware covers common cases, but real-world applications need custom logic. LangChain’s middleware system lets you create custom guardrails using before_agent() and after_agent() hooks.

The before_agent() hook runs before any LLM processing begins. This is perfect for keyword/content filtering, authentication checks, rate limiting, or blocking specific categories of requests.

In [ ]:
from typing import Any
from langchain.agents.middlewarre import(
    AgentMiddlewarre, AgentState, hook_config
)

from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddlewarre):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything --
    zero LLM cost for blocked requests.
    """

    def __ini__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    
    @hook_config(can_jump_to = ["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked -- keyword detected: '{keyword}")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing"
                            "inappropriate content."
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

# Create agent with content filter
filter_agent = create_agent(
    model = "gpt-4o",
    tools = [search_tool],
    middleware = [
        ContentFilterMiddleware(
            banned_keywords = [
                "hack", "exploit", "malware", "jailbreak", "bypass"
            ]
        ),
    ],
)    

Let us break down what is happening in this custom middleware:

AgentMiddleware is the base class for all custom guardrails. You extend it and override the hooks you need.

@hook_config(can_jump_to=["end"]) tells the middleware system that this hook is allowed to short-circuit execution by jumping directly to the end. Without this, you could not block requests.

before_agent() receives the current state (including all messages) and can either return None (let execution continue) or return a new state with "jump_to": "end" (block execution and return immediately).

The beauty of this pattern is that blocked requests never touch the LLM. Zero tokens consumed, zero cost incurred.

## Testing the Content Filter

# Test 1: Safe request -- should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("Safe request response:")
print(result["messages"][-1].content)

In [ ]:
# Test 2: Unsafe request -- should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("Unsafe request response:")
print(result["messages"][-1].content)

The safe request passes through to the LLM normally. The unsafe request gets caught by the keyword filter and returns a canned response without ever hitting the model.